# Tox21 Training Notebook (Colab)

This notebook trains per-assay toxicity models for CodeCure Problem Statement 1.

Planned approach:
- RDKit descriptors + Morgan fingerprints
- KMeans cluster-based train/val/test split
- `RandomForestClassifier` baseline
- `XGBClassifier` as the main boosted model
- class imbalance handling with class weighting first


In [ ]:
# If running in Colab, install the required packages.
%pip install -q rdkit-pypi xgboost imbalanced-learn shap

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, Lipinski, rdMolDescriptors

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

In [ ]:
# Set paths. In Colab, upload the CSV or mount Drive and update DATA_DIR.
DATA_DIR = Path('/content')
TOX21_PATH = DATA_DIR / 'tox21.csv'
OUTPUT_DIR = DATA_DIR / 'outputs'
MODEL_DIR = DATA_DIR / 'models'
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

TARGETS = [
    'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
    'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53'
]

df = pd.read_csv(TOX21_PATH)
df['smiles'] = df['smiles'].astype(str).str.strip()
df.head()

In [ ]:
def mol_from_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    return mol

def morgan_fp(mol, radius=2, n_bits=1024):
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def calc_descriptors(mol):
    return {
        'MolWt': Descriptors.MolWt(mol),
        'MolLogP': Descriptors.MolLogP(mol),
        'TPSA': rdMolDescriptors.CalcTPSA(mol),
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        'RotBonds': Lipinski.NumRotatableBonds(mol),
        'RingCount': Lipinski.RingCount(mol),
        'AromaticRings': Lipinski.NumAromaticRings(mol),
        'FractionCSP3': rdMolDescriptors.CalcFractionCSP3(mol),
    }

def build_feature_matrix(input_df, fp_bits=1024):
    valid_rows = []
    desc_rows = []
    fp_rows = []

    for idx, row in input_df.iterrows():
        mol = mol_from_smiles(row['smiles'])
        if mol is None:
            continue
        valid_rows.append(idx)
        desc_rows.append(calc_descriptors(mol))
        fp_rows.append(morgan_fp(mol, n_bits=fp_bits))

    desc_df = pd.DataFrame(desc_rows, index=valid_rows)
    fp_df = pd.DataFrame(fp_rows, index=valid_rows, columns=[f'fp_{i}' for i in range(fp_bits)])
    features = pd.concat([desc_df, fp_df], axis=1)
    return input_df.loc[valid_rows].copy(), features

df_valid, X = build_feature_matrix(df)
print(df_valid.shape, X.shape)

In [ ]:
# KMeans split on a standardized subset of descriptors + a subset of fingerprint bits.
cluster_input = X.copy()
cluster_input = cluster_input.iloc[:, :128].copy() if cluster_input.shape[1] > 128 else cluster_input.copy()
cluster_input = pd.DataFrame(StandardScaler().fit_transform(cluster_input), index=cluster_input.index)

kmeans = KMeans(n_clusters=20, random_state=42, n_init=10)
clusters = pd.Series(kmeans.fit_predict(cluster_input), index=cluster_input.index, name='cluster')

unique_clusters = sorted(clusters.unique())
rng = np.random.default_rng(42)
rng.shuffle(unique_clusters)

n_clusters = len(unique_clusters)
train_cut = int(0.70 * n_clusters)
val_cut = int(0.85 * n_clusters)

train_clusters = set(unique_clusters[:train_cut])
val_clusters = set(unique_clusters[train_cut:val_cut])
test_clusters = set(unique_clusters[val_cut:])

split_map = {}
for idx, c in clusters.items():
    if c in train_clusters:
        split_map[idx] = 'train'
    elif c in val_clusters:
        split_map[idx] = 'val'
    else:
        split_map[idx] = 'test'

split_series = pd.Series(split_map, name='split')
split_series.value_counts()

In [ ]:
def evaluate_binary(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc': float(average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else np.nan,
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
    }

results = []
selected_models = {}

for target in TARGETS:
    work_df = df_valid[[target]].join(split_series).dropna(subset=[target]).copy()
    idx_train = work_df.index[work_df['split'] == 'train']
    idx_val = work_df.index[work_df['split'] == 'val']
    idx_test = work_df.index[work_df['split'] == 'test']

    X_train = X.loc[idx_train]
    X_val = X.loc[idx_val]
    X_test = X.loc[idx_test]

    y_train = work_df.loc[idx_train, target].astype(int)
    y_val = work_df.loc[idx_val, target].astype(int)
    y_test = work_df.loc[idx_test, target].astype(int)

    pos = max(int(y_train.sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    rf = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
    rf.fit(X_train, y_train)
    rf_val_prob = rf.predict_proba(X_val)[:, 1]
    rf_val_metrics = evaluate_binary(y_val, rf_val_prob)

    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        tree_method='hist',
        device='cuda'
    )
    xgb.fit(X_train, y_train)
    xgb_val_prob = xgb.predict_proba(X_val)[:, 1]
    xgb_val_metrics = evaluate_binary(y_val, xgb_val_prob)

    best_name = 'xgboost' if (xgb_val_metrics['roc_auc'] >= rf_val_metrics['roc_auc']) else 'random_forest'
    best_model = xgb if best_name == 'xgboost' else rf
    test_prob = best_model.predict_proba(X_test)[:, 1]
    test_metrics = evaluate_binary(y_test, test_prob)

    model_path = MODEL_DIR / f'{target}.joblib'
    joblib.dump(best_model, model_path)
    selected_models[target] = {'model': best_name, 'path': str(model_path)}

    results.append({
        'target': target,
        'train_n': int(len(idx_train)),
        'val_n': int(len(idx_val)),
        'test_n': int(len(idx_test)),
        'positive_rate_train': float(y_train.mean()),
        'rf_val_roc_auc': rf_val_metrics['roc_auc'],
        'xgb_val_roc_auc': xgb_val_metrics['roc_auc'],
        'selected_model': best_name,
        'test_roc_auc': test_metrics['roc_auc'],
        'test_pr_auc': test_metrics['pr_auc'],
        'test_f1': test_metrics['f1'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
    })

results_df = pd.DataFrame(results).sort_values('test_roc_auc', ascending=False)
results_df

In [ ]:
results_path = OUTPUT_DIR / 'tox21_model_results.csv'
results_df.to_csv(results_path, index=False)

metadata = {
    'targets': TARGETS,
    'split_strategy': 'kmeans_cluster_split_70_15_15',
    'feature_type': 'rdkit_descriptors_plus_morgan_1024',
    'selected_models': selected_models,
}

with open(MODEL_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print('Saved:', results_path)
results_df[['target', 'selected_model', 'test_roc_auc', 'test_pr_auc', 'test_f1']].head(12)

## Next notebook steps

- add feature importance export
- add SHAP for top-performing targets
- reuse the saved models for ZINC batch screening
- wire the same feature pipeline into Streamlit
